In [1]:
import collections
import io
import json
import pathlib
import os
import sqlite3
import sys
import urllib.parse

import cbor2
import cbor4dns.encode
import polars

COLLECT_JSONS_DIR = pathlib.Path.cwd()

os.environ["COLLECT_JSONS_DIR"] = str(COLLECT_JSONS_DIR)
if str(COLLECT_JSONS_DIR) not in sys.path:
    sys.path.append(str(COLLECT_JSONS_DIR))

In [2]:
%%bash

tmux new-session -s "collect_jsons" -n "collect_jsons_and_dns" -d "INPUT_PATH='/users/lenders/pivot-eval/' ${COLLECT_JSONS_DIR}/collect_jsons_and_dns.sh 2> collect_jsons_and_dns.log"

In [3]:
def convert_to_cbor(obj_json):
    obj = json.loads(obj_json, object_pairs_hook=collections.OrderedDict)
    return cbor2.dumps(obj, canonical=True)

def query_to_json(url):
    url_parts = urllib.parse.urlsplit(url)
    query = urllib.parse.parse_qs(url_parts.query)
    if query:
        for key in query:
            for i, value in enumerate(query[key]):
                try:
                    query[key][i] = int(value)
                except ValueError:
                    try:
                        query[key][i] = float(value)
                    except ValueError:
                        pass
            if len(query[key]) == 1:
                query[key] = query[key][0]
        return json.dumps(query, separators=(',', ':'), sort_keys=True)
    return None

def remove_query(url):
    url_parts = urllib.parse.urlsplit(url)
    assert not url_parts.fragment
    return urllib.parse.urlunsplit(
        (url_parts.scheme, url_parts.netloc, url_parts.path, None, None)
    )

def encode_cbor_dns(msgs):
    if isinstance(msgs, bytes):
        msg = msgs
        orig_query = None
    else:
        msg = msgs["classic_response"]
        orig_query = msgs["cbor_query"]        
    with io.BytesIO() as file:
        encoder = cbor4dns.encode.Encoder(
            file,
            packed=False,
            always_omit_question=False,
        )
        encoder.encode(msg, orig_query)
        return file.getvalue()

df = polars.read_csv(
    COLLECT_JSONS_DIR / "2024-09-01-sample.csv",
    separator=";",
    quote_char="'"
).with_row_index(name="id", offset=1)
objects = df[["id", "url", "http_status", "obj"]].rename({"obj": "json"})
objects[["cbor"]] = objects.select(polars.col("json").map_elements(convert_to_cbor, return_dtype=polars.Binary))
objects[["json_query"]] = objects.select(polars.col("url").map_elements(query_to_json, return_dtype=polars.String))
objects[["url_wo_query"]] = objects.select(polars.col("url").map_elements(remove_query, return_dtype=polars.String))
objects[["cbor_query"]] = objects.select(polars.col("json_query").map_elements(convert_to_cbor, return_dtype=polars.Binary))

dns = polars.concat(
    [
        df.filter(~polars.col("https_q_dns_msg").is_null())[
            [
                "https_q_query_name",
                "https_q_query_type",
                "https_q_response_names",
                "https_q_response_types",
                "https_q_dns_msg",
                "https_r_response_names",
                "https_r_response_types",
                "https_r_dns_msg",
                "url",
                "id",
            ]
        ].rename(
            {
                "https_q_query_name": "name",
                "https_q_query_type": "type",
                "https_q_dns_msg": "classic_query",
                "https_q_response_names": "query_add_names",
                "https_q_response_types": "query_add_types",
                "https_r_dns_msg": "classic_response",
                "https_r_response_names": "response_names",
                "https_r_response_types": "response_types",
            }
        ),
        df.filter(~polars.col("aaaa_q_dns_msg").is_null())[
            [
                "aaaa_q_query_name",
                "aaaa_q_query_type",
                "aaaa_q_response_names",
                "aaaa_q_response_types",
                "aaaa_q_dns_msg",
                "aaaa_r_response_names",
                "aaaa_r_response_types",
                "aaaa_r_dns_msg",
                "url",
                "id",
            ]
        ].rename(
            {
                "aaaa_q_query_name": "name",
                "aaaa_q_query_type": "type",
                "aaaa_q_dns_msg": "classic_query",
                "aaaa_q_response_names": "query_add_names",
                "aaaa_q_response_types": "query_add_types",
                "aaaa_r_dns_msg": "classic_response",
                "aaaa_r_response_names": "response_names",
                "aaaa_r_response_types": "response_types",
            }
        ),
        df.filter(~polars.col("a_q_dns_msg").is_null())[
            [
                "a_q_query_name",
                "a_q_query_type",
                "a_q_response_names",
                "a_q_response_types",
                "a_q_dns_msg",
                "a_r_response_names",
                "a_r_response_types",
                "a_r_dns_msg",
                "url",
                "id",
            ]
        ].rename(
            {
                "a_q_query_name": "name",
                "a_q_query_type": "type",
                "a_q_dns_msg": "classic_query",
                "a_q_response_names": "query_add_names",
                "a_q_response_types": "query_add_types",
                "a_r_dns_msg": "classic_response",
                "a_r_response_names": "response_names",
                "a_r_response_types": "response_types",
            }
        ),
    ]
).sort(["name", "type", "classic_query", "classic_response"]).rename({"id": "obj_id"})
dns[["classic_query"]] = dns.select(polars.col("classic_query").map_elements(bytes.fromhex, return_dtype=polars.Binary))
dns[["classic_response"]] = dns.select(polars.col("classic_response").map_elements(bytes.fromhex, return_dtype=polars.Binary))
dns[["cbor_query"]] = dns.select(polars.col("classic_query").map_elements(encode_cbor_dns, return_dtype=polars.Binary))
dns = dns.with_columns(polars.struct("classic_response", "cbor_query").map_elements(encode_cbor_dns, return_dtype=polars.Binary).alias('cbor_response'))
dns = dns[
    [
        "name",
        "type",
        "query_add_names",
        "query_add_types",
        "classic_query",
        "cbor_query",
        "response_names",
        "response_types",
        "classic_response",
        "cbor_response",
        "url",
        "obj_id",
    ]
]

In [4]:
DB_FILE = COLLECT_JSONS_DIR / "2024-09-01-sample.sqlite3"

with sqlite3.connect(DB_FILE) as conn:
    cur = conn.cursor()
    cur.execute("""CREATE TABLE objects (
        id INTEGER PRIMARY KEY,
        url TEXT,
        http_status INTEGER,
        json TEXT,
        cbor BLOB,
        json_query TEXT,
        url_wo_query TEXT,
        cbor_query BLOB
    );""")
    cur.execute("""CREATE TABLE dns (
        id INTEGER PRIMARY KEY,
        name TEXT,
        type TEXT,
        query_add_names TEXT,
        query_add_types TEXT,
        classic_query BLOB,
        cbor_query BLOB,
        response_names TEXT,
        response_types TEXT,
        classic_response BLOB,
        cbor_response BLOB,
        url TEXT,
        obj_id INTEGER,
        FOREIGN KEY (obj_id) REFERENCES objects(id)
    );""")
    conn.commit()


objects.write_database("objects", f"sqlite:///{DB_FILE}", if_table_exists="append")
dns.write_database("dns", f"sqlite:///{DB_FILE}", if_table_exists="append")

with sqlite3.connect(DB_FILE) as conn:
    cur = conn.cursor()
    cur.execute("CREATE INDEX objects_url ON objects (url);")
    cur.execute("CREATE INDEX objects_url_json_query ON objects (url_wo_query, json_query);")
    cur.execute("CREATE INDEX objects_url_cbor_query ON objects (url_wo_query, cbor_query);")
    cur.execute("CREATE INDEX dns_url ON dns (url);")
    cur.execute("CREATE INDEX dns_name_type ON dns (name, type);")
    conn.commit()